## Clone Flow-Bench and Prepare the Python Environment

In [ ]:
# do NOT run this on your local PC
# Clone Flow-Bench
!git clone https://github.com/Qi-Zhou-Geo/Flow-Bench.git
%cd Flow-Bench

# Install dependencies (pip equivalents)
!pip install -r config/Flow-Bench-requirements.txt

## Importing modules

In [ ]:
#!/usr/bin/python
# -*- coding: UTF-8 -*-

#__modification time__ = 2026-01-20
#__author__ = Qi Zhou, Helmholtz Centre Potsdam - GFZ German Research Centre for Geosciences
#__find me__ = qi.zhou@gfz.de, qi.zhou.geo@gmail.com, https://github.com/Qi-Zhou-Geo
# Please do not distribute this code without the author's permission

from obspy.core import UTCDateTime
from obspy.clients.fdsn import Client

# <editor-fold desc="add the sys.path to search for custom modules">
import sys
from pathlib import Path

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Google Colab - mount drive first
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = Path("/content/drive/MyDrive/Flow-Bench")
else:
    # current_dir = Path(__file__).resolve().parent # Jupyter does not support this
    current_dir = "/Users/qizhou/#python/Flow-Bench/demo"

    # using ".parent" on "pathlib.Path" object moves one level up the directory hierarchy
    # project_root = current_dir.parent.parent # Jupyter does not support this
    project_root = "/Users/qizhou/#python/Flow-Bench"


sys.path.append(str(project_root))
print(f"Project root: {project_root}")
# </editor-fold>


# import the custom functions
from functions.model import FlowBench

## Fetch some seismic signals

In [ ]:
client = Client("GEOFON")

network, station, location, channel = "XN", "NEP08", "", "HHZ"
starttime = UTCDateTime("2016-07-04T10:00:00")
endtime = UTCDateTime("2016-07-06T14:00:00")

inventory = client.get_stations(network=network, station=station, location=location, channel=channel,
                                starttime=starttime, endtime=endtime, level="response")

st = client.get_waveforms(network=network, station=station, location=location, channel=channel,
                          starttime=starttime, endtime=endtime)

st.merge(method=1, fill_value='latest', interpolation_samples=0)
st._cleanup()
st.detrend("linear")
st.detrend("demean")
st.taper(max_percentage=0.05)
st.remove_response(inventory=inventory, output="VEL", pre_filt=(0.5, 1, 30, 45), water_level=60)
st.filter("bandpass", freqmin=1.0, freqmax=25.0)
st.detrend("linear")
st.detrend("demean")
st.taper(max_percentage=0.05)
st.trim(starttime + 2 * 3600, endtime - 2 * 3600)

print(st[0].stats)

## Have Fun with Flow-Bench

In [ ]:
model = FlowBench(model_version=0.3, output_path=project_root)
model.define_event_timing(st)

# use the STA/LTA based timing
event_start, event_end = "2016-07-05T15:16:27", "2016-07-05T17:07:48"
model.freq_domain(st, event_start, event_end)

# use the more data to cover the slient period
event_start, event_end = "2016-07-05T14:00:00", "2016-07-05T19:00:00"
model.time_domain(st, event_start, event_end)
model.plot()